# HOPE — Pipeline Sentiment Tunisien

**Architecture :**
```
Texte entrant
      |
      v
Détecteur de langue (langdetect + règles Arabizi)
      |
      +-- Arabizi / Arabe  ----------------------------> MarBERT --> label
      |
      +-- Français / Anglais -- Traduction (Groq) ----> MarBERT --> label
                                                              |
                                                              v
                                                         LIME (explication)
```
Un seul modèle de sentiment (ton MarBERT fine-tuné). La traduction est juste un pré-traitement.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, re
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from langdetect import detect, LangDetectException
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # lit GROQ_API_KEY depuis .env

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH  = './models/MARBERT-balanced-after-gen-neut'
ID2LABEL    = {0: 'negatif', 1: 'neutre', 2: 'positif'}
LABEL2ID    = {v: k for k, v in ID2LABEL.items()}
CLASS_NAMES = ['negatif', 'neutre', 'positif']
COLORS      = {'negatif': '#d93025', 'neutre': '#e37400', 'positif': '#1e8e3e'}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device.upper()}')
print(f'Model  : {MODEL_PATH}')

In [ ]:
# ── Chargement MarBERT + Groq ────────────────────────────────────────────────
print('Chargement du modele...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
).to(device)
model.eval()

groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))
print('Pret.')

In [ ]:
# ── Detecteur de langue ───────────────────────────────────────────────────────
# Mots typiquement tunisiens qui ne peuvent pas etre confondus avec du francais/anglais
_TUNISIAN = {
    'ya','ma','fi','eli','elli','mel','ken','bch','mta3','weld','bent',
    'yji','nhar','kol','fel','barcha','bahi','taw','chy','shkoun','wesh',
    'sahih','famma','mafamma','aslema','marhba','yasser','ahna','entom',
    'howa','heya','heka','hedhi','houma','nchallah','inchallah','taberaka',
    'mabrouk','saha','brabi','zeda','mouch','barra','haki','kima','chnowa',
    'chniya','winou','fein','leh','sayb','tfeh','barcha','9dar','3leh',
}

def detect_language(text: str) -> str:
    """
    Retourne : 'arabic' | 'arabizi' | 'french' | 'english'
    Defaut : 'arabizi' (si incertain, on envoie direct au MarBERT)
    """
    text = text.strip()
    if not text:
        return 'arabizi'

    # 1. Caracteres arabes => passe direct au MarBERT
    arabic_chars = len(re.findall(r'[\u0600-\u06FF]', text))
    total_alpha  = max(len(re.findall(r'\w', text)), 1)
    if arabic_chars / total_alpha > 0.25:
        return 'arabic'

    # 2. Arabizi : chiffres utilises comme lettres arabes (3=ain, 7=ha, 9=qaf)
    score = 0
    score += len(re.findall(r'[379]', text)) * 2   # marqueurs forts
    score += len(re.findall(r'[48]',  text))         # marqueurs moderes
    mots = set(re.findall(r'[a-zA-Z]+', text.lower()))
    score += len(mots & _TUNISIAN) * 3
    if score >= 3:
        return 'arabizi'

    # 3. langdetect pour le reste
    try:
        lang = detect(text)
        if lang == 'fr': return 'french'
        if lang == 'en': return 'english'
    except LangDetectException:
        pass

    return 'arabizi'  # defaut securise


# --- Test rapide ---
tests = [
    ('aslema ya hmema',                        'arabizi'),
    ('marhba barcha bahi',                     'arabizi'),
    ('sayb a3lik mel lou4a tay7t 9dar',        'arabizi'),
    ('slim yeriya7i',                           'arabizi'),
    ('mabrok 3lik',                             'arabizi'),
    ('\u0645\u0628\u0631\u0648\u0643 \u0639\u0644\u064a\u0643', 'arabic'),
    ('bravo tres juste',                        'french'),
    ('the weather is nice today',               'english'),
]
print('Test detection de langue :')
all_ok = True
for txt, expected in tests:
    got = detect_language(txt)
    ok  = got == expected
    if not ok: all_ok = False
    print(f'  {chr(10003) if ok else chr(10007)}  [{got:8s}]  "{txt}"')
print('Tout bon!' if all_ok else 'Certains cas incorrects — verifier.')

In [ ]:
# ── Traduction vers l'arabe (Groq) ───────────────────────────────────────────
_PROMPT = """Traduis ce texte en arabe. Reponds uniquement avec la traduction arabe, sans explication.

Texte : {text}
Traduction arabe :""
def translate_to_arabic(text: str, source_lang: str) -> str:
    """Traduit un texte francais ou anglais en arabe via Groq Llama."""
    try:
        resp = groq_client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role': 'user', 'content': _PROMPT.format(text=text)}],
            max_tokens=256,
            temperature=0,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f'  [Traduction echouee : {e}] => texte original envoye au MarBERT')
        return text  # fallback : MarBERT fait de son mieux


# --- Test ---
print('Test traduction :')
print('FR :', translate_to_arabic('bravo tres juste', 'french'))
print('EN :', translate_to_arabic('the weather is supposedly good', 'english'))

In [ ]:
# ── Prediction MarBERT ────────────────────────────────────────────────────────
def predict_proba(texts) -> np.ndarray:
    """Retourne tableau (N, 3) : colonnes negatif / neutre / positif."""
    enc = tokenizer(
        list(texts), return_tensors='pt',
        padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1).cpu().numpy()


# ── Pipeline complet ──────────────────────────────────────────────────────────
def run_pipeline(text: str, verbose: bool = True):
    """
    1. Detecte la langue
    2. Traduit si francais/anglais
    3. Envoie au MarBERT
    Retourne : (label, confiance, probas, langue_detectee, texte_envoye_au_modele)
    """
    lang       = detect_language(text)
    translated = text

    if lang in ('french', 'english'):
        if verbose: print(f'  Detecte : {lang} => traduction en cours...')
        translated = translate_to_arabic(text, lang)
        if verbose: print(f'  Traduit : "{translated}"')
    else:
        if verbose: print(f'  Detecte : {lang} => envoi direct au MarBERT')

    probs   = predict_proba([translated])[0]
    pred_id = int(np.argmax(probs))
    pred    = ID2LABEL[pred_id]
    conf    = float(probs[pred_id])
    return pred, conf, probs, lang, translated


# --- Verification rapide ---
print('Verification pipeline :')
for t in ['aslema ya hmema', 'bravo tres juste', 'the weather is nice']:
    pred, conf, probs, lang, _ = run_pipeline(t, verbose=False)
    print(f'  [{lang:8s}]  "{t}"  =>  {pred.upper()} ({conf*100:.1f}%)')


# ── LIME ──────────────────────────────────────────────────────────────────────
explainer = LimeTextExplainer(class_names=CLASS_NAMES, bow=False)
print('\nLIME pret.')

---
## Demo interactive
Modifie `TEXT` dans la cellule suivante et appuie sur **Shift+Enter** sur les deux cellules.

In [ ]:
# =====================================================================
#  CHANGE CE TEXTE, puis Shift+Enter sur cette cellule ET la suivante
TEXT = "aslema ya hmema"
# =====================================================================

pred, conf, probs, lang, text_for_marbert = run_pipeline(TEXT)

print(f'\n  Texte original   : "{TEXT}"')
if text_for_marbert != TEXT:
    print(f'  Envoye au modele : "{text_for_marbert}"')
print(f'  Prediction       : {pred.upper()}  ({conf*100:.1f}% de confiance)')
print(f'  negatif={probs[0]:.3f}  neutre={probs[1]:.3f}  positif={probs[2]:.3f}')

if conf < 0.70:
    print(f'  !! Confiance faible — le modele est incertain')

# --- Graphe probabilites ---
fig, ax = plt.subplots(figsize=(6, 2.5))
fig.patch.set_facecolor('#fafafa')
bars = ax.barh(
    CLASS_NAMES, [probs[0], probs[1], probs[2]],
    color=[COLORS[c] for c in CLASS_NAMES], edgecolor='white', height=0.5
)
for bar, p in zip(bars, [probs[0], probs[1], probs[2]]):
    ax.text(p + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{p*100:.1f}%', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.25)
ax.axvline(1/3, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
note = f' (traduit depuis {lang})' if lang in ('french', 'english') else f' ({lang})'
ax.set_title(f'{pred.upper()} — {conf*100:.1f}%{note}',
             fontsize=12, fontweight='bold', color=COLORS[pred])
ax.set_facecolor('white')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── LIME : pourquoi ce resultat ? (~15 secondes) ─────────────────────────────
pred_id = int(np.argmax(probs))

exp = explainer.explain_instance(
    text_for_marbert, predict_proba,
    num_features=10, num_samples=600, labels=[0, 1, 2]
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#fafafa')
short = text_for_marbert[:65] + ('...' if len(text_for_marbert) > 65 else '')
fig.suptitle(
    f'LIME — "{short}"\nPrediction : {pred.upper()} ({conf*100:.1f}%)',
    fontsize=12, fontweight='bold', color=COLORS[pred], y=1.02
)

for ax, class_id in zip(axes, [0, 1, 2]):
    class_lbl = ID2LABEL[class_id]
    color     = COLORS[class_lbl]
    weights   = exp.as_list(label=class_id)
    if not weights:
        ax.text(0.5, 0.5, 'Aucun feature', ha='center', va='center')
        continue
    words  = [w[0] for w in weights]
    values = [w[1] for w in weights]
    order  = np.argsort(np.abs(values))[::-1]
    words  = [words[i] for i in order]
    values = [values[i] for i in order]
    bar_colors = [color if v > 0 else '#9e9e9e' for v in values]
    ax.barh(range(len(words)), values, color=bar_colors, edgecolor='white', height=0.65)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Importance  (+ => pousse vers cette classe)', fontsize=9)
    marker = '  <= PREDIT' if class_id == pred_id else ''
    ax.set_title(f'{class_lbl.upper()}  p={probs[class_id]:.3f}{marker}',
                 fontsize=11, fontweight='bold', color=color)
    ax.grid(True, axis='x', alpha=0.3)
    ax.set_facecolor('white')
    ax.legend(handles=[
        mpatches.Patch(color=color,      label=f'=> {class_lbl}'),
        mpatches.Patch(color='#9e9e9e',  label='contre'),
    ], fontsize=8)

plt.tight_layout()
plt.show()

print(f'\nMots qui ont le plus influence => {pred.upper()}')
for word, score in sorted(exp.as_list(label=pred_id), key=lambda x: -abs(x[1]))[:8]:
    arrow = 'POUR    ' if score > 0 else 'CONTRE  '
    print(f'  {arrow} {pred:8s}  |  "{word}"  ({score:+.4f})')

---
## Batch — tester plusieurs textes d'un coup (sans LIME, instantane)

In [ ]:
# ── Batch prediction ──────────────────────────────────────────────────────────
batch_texts = [
    'aslema ya hmema',
    'sayb a3lik mel lou4a tay7t 9dar',
    'the weather supposedly is good but we do not know that yet',
    'amali abone',
    'chbik khra',
    'salem',
    'bravo tres juste',
    # <== ajoute tes propres textes ici
]

rows = []
for text in batch_texts:
    pred, conf, probs, lang, translated = run_pipeline(text, verbose=False)
    rows.append({
        'texte'       : text[:50] + ('...' if len(text) > 50 else ''),
        'langue'      : lang,
        'prediction'  : pred,
        'confiance'   : f'{conf*100:.1f}%',
        'p(neg)'      : f'{probs[0]:.3f}',
        'p(neu)'      : f'{probs[1]:.3f}',
        'p(pos)'      : f'{probs[2]:.3f}',
        'traduit_en'  : '' if translated == text else translated[:40],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))